# SF Census Tract Matcher

Resolves a single census tract per row from `4_sf_zipcode_matcher.csv`.

Logic:
- **One-to-One** rows: use the primary `census_tract` directly.
- **One-to-Many (Tie)** rows: extract the 6-digit tract from each entry in `tie_geoids`.
    - First, try the subset of ties that matched on ZIP. If their tracts all agree, that single tract is the resolved value.
    - If the ZIP-matching subset is empty (e.g. `One-to-Many Matched No Output`) or disagrees, fall back to checking *all* ties: if every tie shares a tract, use it.
    - If neither rule resolves to a single tract, the row is flagged as `Tract Ambiguous`.

Output: `data/1_derived/5_sf_census_tract_matcher.csv` and a markdown report under `output/2_analysis/tables/`.

In [1]:
import os
import re
import pandas as pd
from IPython.display import Markdown, display

in_path = "../../data/1_derived/4_sf_zipcode_matcher.csv"
out_path = "../../data/1_derived/5_sf_census_tract_matcher.csv"
report_path = "../../output/2_analysis/tables/5_sf_census_tract_matcher_report.md"

df = pd.read_csv(in_path)
print(f"Loaded {len(df):,} rows from {in_path}")

Loaded 9,352 rows from ../../data/1_derived/4_sf_zipcode_matcher.csv


In [2]:
def parse_pipe_list(value):
    if pd.isna(value):
        return []
    return [p.strip() for p in str(value).split("|") if p.strip()]


def normalize_tract(value):
    """Return zero-padded 6-digit census tract string, or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0", s):
        s = s.split(".")[0]
    if not s.isdigit():
        return None
    return s.zfill(6)


def normalize_zip5(value):
    """Return 5-digit ZIP string, or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0", s):
        s = s.split(".")[0]
    m = re.search(r"(\d{5})", s)
    return m.group(1) if m else None


def extract_tract_from_geoid(geoid):
    """Extract the 6-digit census tract from a 15-digit GEOID string.

    Format: state(2) + county(3) + tract(6) + block(4).
    """
    if geoid is None or (isinstance(geoid, float) and pd.isna(geoid)):
        return None
    s = re.sub(r"\D", "", str(geoid))
    if len(s) < 11:
        return None
    s = s.zfill(15)
    return s[5:11]


def extract_tracts_from_tie_geoids(tie_geoids):
    """Return list of 6-digit tract strings from pipe-delimited tie_geoids."""
    out = []
    for token in parse_pipe_list(tie_geoids):
        t = extract_tract_from_geoid(token)
        if t is not None:
            out.append(t)
    return out


def get_zip_matching_tie_indices(row):
    """Indices of tie outputs whose ZIP matches the original ZIP."""
    original_zip = normalize_zip5(row.get("zip_original"))
    if original_zip is None:
        return []
    addresses = parse_pipe_list(row.get("tie_matched_addresses"))
    indices = []
    for i, addr in enumerate(addresses):
        m_end = re.search(r"(\d{5})(?:-\d{4})?\s*$", addr)
        z = m_end.group(1) if m_end else None
        if z is None:
            m_any = re.search(r"(\d{5})", addr)
            z = m_any.group(1) if m_any else None
        if z == original_zip:
            indices.append(i)
    return indices

In [3]:
def resolve_row(row):
    match_type = row.get("match_type")
    primary_tract = normalize_tract(row.get("census_tract"))
    tie_tracts_all = extract_tracts_from_tie_geoids(row.get("tie_geoids"))
    zip_idx = get_zip_matching_tie_indices(row)
    tie_tracts_zip = [tie_tracts_all[i] for i in zip_idx if i < len(tie_tracts_all)]

    unique_tie_all = sorted(set(tie_tracts_all))
    unique_tie_zip = sorted(set(tie_tracts_zip))

    tract_consistent_in_zip_matches = (
        len(unique_tie_zip) == 1 if len(tie_tracts_zip) > 0 else None
    )
    tract_consistent_in_all_ties = (
        len(unique_tie_all) == 1 if len(tie_tracts_all) > 0 else None
    )

    if match_type == "One-to-One":
        if primary_tract is not None:
            method = "One-to-One Primary"
            final_tract = primary_tract
        else:
            method = "Tract Missing"
            final_tract = None
    else:
        if len(tie_tracts_zip) > 0 and len(unique_tie_zip) == 1:
            method = "Tie Resolved via ZIP-Matching Ties"
            final_tract = unique_tie_zip[0]
        elif len(tie_tracts_all) > 0 and len(unique_tie_all) == 1:
            method = "Tie Resolved via All Ties Unanimous"
            final_tract = unique_tie_all[0]
        elif len(unique_tie_zip) > 1 or (len(unique_tie_zip) == 0 and len(unique_tie_all) > 1):
            method = "Tract Ambiguous"
            final_tract = None
        else:
            method = "Tract Missing"
            final_tract = None

    return pd.Series({
        "primary_tract": primary_tract,
        "tie_tracts_all_str": " | ".join(tie_tracts_all) if tie_tracts_all else None,
        "tie_tracts_zip_str": " | ".join(tie_tracts_zip) if tie_tracts_zip else None,
        "tie_tract_unique_count_all": len(unique_tie_all),
        "tie_tract_unique_count_zip": len(unique_tie_zip),
        "tract_consistent_in_zip_matches": tract_consistent_in_zip_matches,
        "tract_consistent_in_all_ties": tract_consistent_in_all_ties,
        "tract_resolution_method": method,
        "final_census_tract": final_tract,
    })


resolved = df.apply(resolve_row, axis=1)
df = pd.concat([df, resolved], axis=1)
df["tract_resolved"] = df["final_census_tract"].notna()
df["tract_flag"] = df["tract_resolution_method"].isin(["Tract Ambiguous", "Tract Missing"])

print("tract_resolution_method counts:")
print(df["tract_resolution_method"].value_counts(dropna=False))

tract_resolution_method counts:
tract_resolution_method
One-to-One Primary                    8980
Tract Missing                          354
Tract Ambiguous                         13
Tie Resolved via ZIP-Matching Ties       5
Name: count, dtype: int64


In [4]:
tie_rows = df[df["match_type"] == "One-to-Many"].copy()
n_tie = len(tie_rows)
n_tie_zip_consistent = int((tie_rows["tract_consistent_in_zip_matches"] == True).sum())
n_tie_all_consistent = int((tie_rows["tract_consistent_in_all_ties"] == True).sum())
n_tie_resolved = int(tie_rows["tract_resolved"].sum())
n_tie_ambiguous = int((tie_rows["tract_resolution_method"] == "Tract Ambiguous").sum())
n_tie_missing = int((tie_rows["tract_resolution_method"] == "Tract Missing").sum())

print(f"Total One-to-Many (tie) rows:                                {n_tie}")
print(f"  Tie rows where all ZIP-matching ties share same tract:     {n_tie_zip_consistent}")
print(f"  Tie rows where ALL ties share same tract:                  {n_tie_all_consistent}")
print(f"  Tie rows resolved to a single census tract (final):        {n_tie_resolved}")
print(f"  Tie rows ambiguous (multiple distinct tracts):             {n_tie_ambiguous}")
print(f"  Tie rows missing tracts entirely:                          {n_tie_missing}")

tie_breakdown = (
    tie_rows.groupby(["zip_match_detail", "tract_resolution_method"], dropna=False)
            .size()
            .reset_index(name="rows")
            .sort_values(["zip_match_detail", "rows"], ascending=[True, False])
)
tie_breakdown

Total One-to-Many (tie) rows:                                18
  Tie rows where all ZIP-matching ties share same tract:     5
  Tie rows where ALL ties share same tract:                  5
  Tie rows resolved to a single census tract (final):        5
  Tie rows ambiguous (multiple distinct tracts):             13
  Tie rows missing tracts entirely:                          0


,zip_match_detail,tract_resolution_method,rows
1,One-to-Many Matched Multiple Outputs,Tract Ambiguous,7
0,One-to-Many Matched Multiple Outputs,Tie Resolved via ZIP-Matching Ties,5
2,One-to-Many Matched No Output,Tract Ambiguous,6


In [5]:
tie_preview = tie_rows[[
    "PropertyId",
    "Address.PostalCode",
    "matched_address",
    "tie_matched_addresses",
    "tie_geoids",
    "primary_tract",
    "tie_tracts_all_str",
    "tie_tracts_zip_str",
    "tie_tract_unique_count_all",
    "tie_tract_unique_count_zip",
    "tract_consistent_in_zip_matches",
    "tract_consistent_in_all_ties",
    "tract_resolution_method",
    "final_census_tract",
]]
tie_preview

,PropertyId,Address.PostalCode,matched_address,tie_matched_addresses,tie_geoids,primary_tract,tie_tracts_all_str,tie_tracts_zip_str,tie_tract_unique_count_all,tie_tract_unique_count_zip,tract_consistent_in_zip_matches,tract_consistent_in_all_ties,tract_resolution_method,final_census_tract
972,321580.0,94124.0,"1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124","1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 ...",060750233002003 | 060750233002003,023300,023300 | 023300,023300 | 023300,1,1,True,True,Tie Resolved via ZIP-Matching Ties,023300
2801,333689.0,94403.0,"3130 LA SELVA CIR, SAN MATEO, CA, 94403","3130 LA SELVA CIR, SAN MATEO, CA, 94403 | 3130...",060816084003008 | 060816084003001,608400,608400 | 608400,608400 | 608400,1,1,True,True,Tie Resolved via ZIP-Matching Ties,608400
3829,365252.0,94105.0,"26 PIER, SAN FRANCISCO, CA, 94111","26 PIER, SAN FRANCISCO, CA, 94111 | 26 PIER, S...",060750105002002 | 060759809001017,010500,010500 | 980900,None,2,0,None,False,Tract Ambiguous,None
4602,783773.0,94401.0,"256 S EL CAMINO REAL, SAN MATEO, CA, 94401","256 S EL CAMINO REAL, SAN MATEO, CA, 94401 | 2...",060816064005002 | 060816059022004,606400,606400 | 605902,606400 | 605902,2,2,False,False,Tract Ambiguous,None
4794,845493.0,94015.0,"100 SKYLINE BLVD, DALY CITY, CA, 94015","100 SKYLINE BLVD, DALY CITY, CA, 94015 | 100 S...",060816015011000 | 060816010004000,601501,601501 | 601000,601501 | 601000,2,2,False,False,Tract Ambiguous,None
5132,1352787.0,94105.0,"5 PACIFIC AVE, SAN FRANCISCO, CA, 94111","5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 | 5 PA...",060750105002006 | 060750108002001,010500,010500 | 010800,None,2,0,None,False,Tract Ambiguous,None
6102,4318893.0,94133.0,"435 BROADWAY ST, SAN FRANCISCO, CA, 94133","435 BROADWAY ST, SAN FRANCISCO, CA, 94133 | 43...",060750106002006 | 060750106002006,010600,010600 | 010600,010600 | 010600,1,1,True,True,Tie Resolved via ZIP-Matching Ties,010600
6110,4327803.0,94005.0,"88 S HILL DR, BRISBANE, CA, 94005","88 S HILL DR, BRISBANE, CA, 94005 | 88 W HILL ...",060816001002017 | 060816001002013,600100,600100 | 600100,600100 | 600100,1,1,True,True,Tie Resolved via ZIP-Matching Ties,600100
6237,4570159.0,94401.0,"313 S SAN MATEO DR, SAN MATEO, CA, 94401","313 S SAN MATEO DR, SAN MATEO, CA, 94401 | 313...",060816063003022 | 060816059021003,606300,606300 | 605902,606300 | 605902,2,2,False,False,Tract Ambiguous,None
6359,5027545.0,94066.0,"751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066","751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 75...",060816040001002 | 060816042001004,604000,604000 | 604200,604000 | 604200,2,2,False,False,Tract Ambiguous,None


In [6]:
n_total = len(df)
n_resolved = int(df["tract_resolved"].sum())
n_flag = int(df["tract_flag"].sum())

method_counts = df["tract_resolution_method"].value_counts(dropna=False).rename_axis("tract_resolution_method").reset_index(name="rows")
method_counts["pct"] = (method_counts["rows"] / n_total * 100).round(2)

print(f"Total rows:               {n_total:,}")
print(f"Resolved to single tract: {n_resolved:,} ({n_resolved/n_total*100:.2f}%)")
print(f"Flagged (ambiguous/miss): {n_flag:,} ({n_flag/n_total*100:.2f}%)")
print()
method_counts

Total rows:               9,352
Resolved to single tract: 8,985 (96.08%)
Flagged (ambiguous/miss): 367 (3.92%)



,tract_resolution_method,rows,pct
0,One-to-One Primary,8980,96.02
1,Tract Missing,354,3.79
2,Tract Ambiguous,13,0.14
3,Tie Resolved via ZIP-Matching Ties,5,0.05


In [7]:
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")

Saved 9,352 rows to ../../data/1_derived/5_sf_census_tract_matcher.csv


In [8]:
def dataframe_to_md_table(frame):
    if frame.empty:
        return "_No rows to display._"
    cols = [str(c) for c in frame.columns]
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = []
    for _, row in frame.iterrows():
        vals = []
        for c in frame.columns:
            v = row[c]
            vals.append("" if pd.isna(v) else str(v).replace("|", "\\|"))
        rows.append("| " + " | ".join(vals) + " |")
    return "\n".join([header, sep] + rows)


report_lines = [
    "# SF Census Tract Match Report",
    "",
    f"- Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"- Input file: {in_path}",
    f"- Output file: {out_path}",
    f"- Total rows: {n_total:,}",
    f"- Rows resolved to a single census tract: {n_resolved:,} ({n_resolved/n_total*100:.2f}%)",
    f"- Rows flagged (ambiguous or missing): {n_flag:,} ({n_flag/n_total*100:.2f}%)",
    "",
    "## Tract Resolution Method Counts",
    dataframe_to_md_table(method_counts),
    "",
    "## Tie Rows: Tract Resolution by ZIP Match Detail",
    f"Total One-to-Many (tie) rows: {n_tie}.",
    "",
    f"- Tie rows whose ZIP-matching ties all share the same tract: **{n_tie_zip_consistent}**",
    f"- Tie rows where every tie (regardless of ZIP) shares the same tract: **{n_tie_all_consistent}**",
    f"- Tie rows resolved to a single tract (final): **{n_tie_resolved}**",
    f"- Tie rows ambiguous (multiple distinct tracts): **{n_tie_ambiguous}**",
    f"- Tie rows missing tract data: **{n_tie_missing}**",
    "",
    dataframe_to_md_table(tie_breakdown),
    "",
    "## All Tie Rows (Detail)",
    dataframe_to_md_table(tie_preview),
]

report_md = "\n".join(report_lines)
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

display(Markdown(report_md))
print(f"\nSaved markdown report to {report_path}")

# SF Census Tract Match Report

- Generated: 2026-05-05 01:12:57
- Input file: ../../data/1_derived/4_sf_zipcode_matcher.csv
- Output file: ../../data/1_derived/5_sf_census_tract_matcher.csv
- Total rows: 9,352
- Rows resolved to a single census tract: 8,985 (96.08%)
- Rows flagged (ambiguous or missing): 367 (3.92%)

## Tract Resolution Method Counts
| tract_resolution_method | rows | pct |
| --- | --- | --- |
| One-to-One Primary | 8980 | 96.02 |
| Tract Missing | 354 | 3.79 |
| Tract Ambiguous | 13 | 0.14 |
| Tie Resolved via ZIP-Matching Ties | 5 | 0.05 |

## Tie Rows: Tract Resolution by ZIP Match Detail
Total One-to-Many (tie) rows: 18.

- Tie rows whose ZIP-matching ties all share the same tract: **5**
- Tie rows where every tie (regardless of ZIP) shares the same tract: **5**
- Tie rows resolved to a single tract (final): **5**
- Tie rows ambiguous (multiple distinct tracts): **13**
- Tie rows missing tract data: **0**

| zip_match_detail | tract_resolution_method | rows |
| --- | --- | --- |
| One-to-Many Matched Multiple Outputs | Tract Ambiguous | 7 |
| One-to-Many Matched Multiple Outputs | Tie Resolved via ZIP-Matching Ties | 5 |
| One-to-Many Matched No Output | Tract Ambiguous | 6 |

## All Tie Rows (Detail)
| PropertyId | Address.PostalCode | matched_address | tie_matched_addresses | tie_geoids | primary_tract | tie_tracts_all_str | tie_tracts_zip_str | tie_tract_unique_count_all | tie_tract_unique_count_zip | tract_consistent_in_zip_matches | tract_consistent_in_all_ties | tract_resolution_method | final_census_tract |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 321580.0 | 94124.0 | 1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 | 1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 \| 1485 BAYSHORE BLVD, SAN FRANCISCO, CA, 94124 | 060750233002003 \| 060750233002003 | 023300 | 023300 \| 023300 | 023300 \| 023300 | 1 | 1 | True | True | Tie Resolved via ZIP-Matching Ties | 023300 |
| 333689.0 | 94403.0 | 3130 LA SELVA CIR, SAN MATEO, CA, 94403 | 3130 LA SELVA CIR, SAN MATEO, CA, 94403 \| 3130 LA SELVA ST, SAN MATEO, CA, 94403 | 060816084003008 \| 060816084003001 | 608400 | 608400 \| 608400 | 608400 \| 608400 | 1 | 1 | True | True | Tie Resolved via ZIP-Matching Ties | 608400 |
| 365252.0 | 94105.0 | 26 PIER, SAN FRANCISCO, CA, 94111 | 26 PIER, SAN FRANCISCO, CA, 94111 \| 26 PIER, SAN FRANCISCO, CA, 94124 | 060750105002002 \| 060759809001017 | 010500 | 010500 \| 980900 |  | 2 | 0 |  | False | Tract Ambiguous |  |
| 783773.0 | 94401.0 | 256 S EL CAMINO REAL, SAN MATEO, CA, 94401 | 256 S EL CAMINO REAL, SAN MATEO, CA, 94401 \| 256 N EL CAMINO REAL, SAN MATEO, CA, 94401 | 060816064005002 \| 060816059022004 | 606400 | 606400 \| 605902 | 606400 \| 605902 | 2 | 2 | False | False | Tract Ambiguous |  |
| 845493.0 | 94015.0 | 100 SKYLINE BLVD, DALY CITY, CA, 94015 | 100 SKYLINE BLVD, DALY CITY, CA, 94015 \| 100 SKYLINE DR, DALY CITY, CA, 94015 | 060816015011000 \| 060816010004000 | 601501 | 601501 \| 601000 | 601501 \| 601000 | 2 | 2 | False | False | Tract Ambiguous |  |
| 1352787.0 | 94105.0 | 5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 | 5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 \| 5 PACIFIC AVE, SAN FRANCISCO, CA, 94133 | 060750105002006 \| 060750108002001 | 010500 | 010500 \| 010800 |  | 2 | 0 |  | False | Tract Ambiguous |  |
| 4318893.0 | 94133.0 | 435 BROADWAY ST, SAN FRANCISCO, CA, 94133 | 435 BROADWAY ST, SAN FRANCISCO, CA, 94133 \| 435 BROADWAY, SAN FRANCISCO, CA, 94133 | 060750106002006 \| 060750106002006 | 010600 | 010600 \| 010600 | 010600 \| 010600 | 1 | 1 | True | True | Tie Resolved via ZIP-Matching Ties | 010600 |
| 4327803.0 | 94005.0 | 88 S HILL DR, BRISBANE, CA, 94005 | 88 S HILL DR, BRISBANE, CA, 94005 \| 88 W HILL DR, BRISBANE, CA, 94005 | 060816001002017 \| 060816001002013 | 600100 | 600100 \| 600100 | 600100 \| 600100 | 1 | 1 | True | True | Tie Resolved via ZIP-Matching Ties | 600100 |
| 4570159.0 | 94401.0 | 313 S SAN MATEO DR, SAN MATEO, CA, 94401 | 313 S SAN MATEO DR, SAN MATEO, CA, 94401 \| 313 N SAN MATEO DR, SAN MATEO, CA, 94401 | 060816063003022 \| 060816059021003 | 606300 | 606300 \| 605902 | 606300 \| 605902 | 2 | 2 | False | False | Tract Ambiguous |  |
| 5027545.0 | 94066.0 | 751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 751 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 060816040001002 \| 060816042001004 | 604000 | 604000 \| 604200 | 604000 \| 604200 | 2 | 2 | False | False | Tract Ambiguous |  |
| 5034363.0 | 94066.0 | 777 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 777 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 777 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 060816040001002 \| 060816042001004 | 604000 | 604000 \| 604200 | 604000 \| 604200 | 2 | 2 | False | False | Tract Ambiguous |  |
| 5344465.0 | 94066.0 | 675 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 675 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 675 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 060816041042006 \| 060816042001014 | 604104 | 604104 \| 604200 | 604104 \| 604200 | 2 | 2 | False | False | Tract Ambiguous |  |
| 6395899.0 | 94015.0 | 31 SKYLINE BLVD, DALY CITY, CA, 94015 | 31 SKYLINE BLVD, DALY CITY, CA, 94015 \| 31 SKYLINE DR, DALY CITY, CA, 94015 | 060816015011000 \| 060816010001000 | 601501 | 601501 \| 601000 | 601501 \| 601000 | 2 | 2 | False | False | Tract Ambiguous |  |
| 8919922.0 | 94107.0 | 70 PIER, SAN FRANCISCO, CA, 94111 | 70 PIER, SAN FRANCISCO, CA, 94111 \| 70 PIER, SAN FRANCISCO, CA, 94124 | 060750105002002 \| 060759809001017 | 010500 | 010500 \| 980900 |  | 2 | 0 |  | False | Tract Ambiguous |  |
| 8919951.0 | 94107.0 | 70 PIER, SAN FRANCISCO, CA, 94111 | 70 PIER, SAN FRANCISCO, CA, 94111 \| 70 PIER, SAN FRANCISCO, CA, 94124 | 060750105002002 \| 060759809001017 | 010500 | 010500 \| 980900 |  | 2 | 0 |  | False | Tract Ambiguous |  |
| 8920002.0 | 94107.0 | 70 PIER, SAN FRANCISCO, CA, 94111 | 70 PIER, SAN FRANCISCO, CA, 94111 \| 70 PIER, SAN FRANCISCO, CA, 94124 | 060750105002002 \| 060759809001017 | 010500 | 010500 \| 980900 |  | 2 | 0 |  | False | Tract Ambiguous |  |
| 8930622.0 | 94063.0 | 499 SEAPORT BLVD, REDWOOD CITY, CA, 94063 | 499 SEAPORT BLVD, REDWOOD CITY, CA, 94063 \| 499 SEAPORT CT, REDWOOD CITY, CA, 94063 | 060816103021005 \| 060816103021005 | 610302 | 610302 \| 610302 | 610302 \| 610302 | 1 | 1 | True | True | Tie Resolved via ZIP-Matching Ties | 610302 |
| 9506693.0 | 94107.0 | 70 PIER, SAN FRANCISCO, CA, 94111 | 70 PIER, SAN FRANCISCO, CA, 94111 \| 70 PIER, SAN FRANCISCO, CA, 94124 | 060750105002002 \| 060759809001017 | 010500 | 010500 \| 980900 |  | 2 | 0 |  | False | Tract Ambiguous |  |


Saved markdown report to ../../output/2_analysis/tables/5_sf_census_tract_matcher_report.md
